In [7]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd
from apt_detection.device_telemetry.features import operation_counts

INT = Path("../../data/interim/device_telemetry")
proc2 = pd.read_parquet(INT / "phase2_process_table.parquet")
ev2 = pd.read_parquet(INT / "phase2_events.parquet")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
X_ops = operation_counts(ev2, proc2)
print(X_ops.shape)
print(list(X_ops.columns))
X_ops.join(proc2["label"]).groupby("label").mean().round(2).T

(12443, 12)
['n_chmod', 'n_connect', 'n_execve', 'n_load', 'n_open', 'n_rename_read', 'n_rename_write', 'n_setgid', 'n_setuid', 'n_unlink', 'n_update', 'n_events']


label,False,True
n_chmod,0.00,0.00
n_connect,2.04,0.00
n_execve,0.96,1.00
n_load,2.12,2.00
n_open,0.04,0.00
n_rename_read,1.59,0.01
n_rename_write,1.59,0.01
n_setgid,0.10,0.00
n_setuid,0.06,0.00
n_unlink,12.94,0.00


In [9]:
from apt_detection.device_telemetry.features import touch_features
X_touch = touch_features(ev2, proc2)
print(X_touch.shape)
(X_touch > 0).join(proc2["label"]).groupby("label").mean().round(3).T

(12443, 9)


label,False,True
n_path_sensitive,0.029,0.000
n_path_persistence,0.011,0.000
n_path_temp,0.018,0.001
n_path_home,0.003,0.001
n_path_hidden,0.004,0.001
n_path_system_bin,0.952,0.999
n_dest_distinct,0.011,0.000
n_dest_external,0.005,0.000
n_ports_distinct,0.011,0.000


In [10]:
from apt_detection.device_telemetry.features import identity_features
X_id = identity_features(proc2)
print(X_id.shape)
X_id.join(proc2["label"]).groupby("label").mean().round(3).T

(12443, 14)


label,False,True
is_root,0.679,0.000
uid_ne_euid,0.004,0.000
prog_shell,0.033,0.009
prog_interpreter,0.004,0.000
prog_recon,0.071,0.001
prog_archive,0.003,0.001
prog_net_tool,0.000,0.001
prog_file_op,0.039,0.980
prog_user_mgmt,0.000,0.000
cmd_len,59.288,78.675


In [11]:
from apt_detection.device_telemetry.features import build_baseline, rarity_features

proc1 = pd.read_parquet(INT / "phase1_process_table.parquet")
baseline = build_baseline(proc1)
print("baseline programs:", len(baseline["counts"]), " baseline processes:", baseline["total"])

X_rare = rarity_features(proc2, baseline)
print(X_rare.join(proc2["label"]).groupby("label").mean().round(3))

new_progs = proc2.loc[X_rare["prog_in_baseline"] == 0]
print("\nPhase 2 programs never seen in Phase 1 (normal vs attack):")
print(new_progs.groupby("label")["name"].value_counts().groupby(level=0).head(8))

baseline programs: 251  baseline processes: 4941
       prog_in_baseline  prog_rarity
label                               
False             0.983        1.808
True              0.992        2.547

Phase 2 programs never seen in Phase 1 (normal vs attack):
label  name           
False  update-alternat    68
       openjdk-11-jre-     3
       openjdk-11-jre:     2
       openjdk-11-jdk-     2
       openjdk-11-jdk:     2
       sd-resolve          1
       shuf                1
       fstrim              1
True   base64             45
       curl                4
       ssh                 3
       file_search.sh      2
       hostname            1
       useradd             1
       arp                 1
       scp                 1
Name: count, dtype: int64


In [13]:
from apt_detection.device_telemetry.features import (
    link_parents, parent_child_pairs, family_features)

baseline_pairs = set(parent_child_pairs(proc1, link_parents(proc1)))
print("parent>child pairs in Phase 1:", len(baseline_pairs))

X_fam = family_features(proc2, baseline_pairs)
print(X_fam.join(proc2["label"]).groupby("label").mean().round(3).T)

parent>child pairs in Phase 1: 401
label            False  True 
n_children       0.780  0.007
parent_is_shell  0.022  0.006
pair_unusual     0.011  0.007
parent_known     0.780  0.007
depth            1.516  0.009


In [18]:
flood = proc2["name"].isin(["cp", "base64"]) & proc2["label"].astype(bool)

groups = pd.Series("normal", index=proc2.index)
groups.loc[proc2["label"].astype(bool)] = "attack: other"
groups.loc[flood] = "attack: cp/base64 flood"

X_ctx.groupby(groups).median().T

,attack: cp/base64 flood,attack: other,normal
ctx_procs_1m,1437.0,2.0,56.0
ctx_new_1m,2.0,1.0,0.0
ctx_procs_5m,1743.5,3.0,93.0
ctx_new_5m,2.0,1.0,0.0
ctx_procs_15m,1749.0,7.0,111.0
ctx_new_15m,2.0,1.0,0.0


In [20]:
from apt_detection.device_telemetry.features import build_feature_table, main_features, FEATURE_GROUPS

F2 = build_feature_table(proc2, ev2, baseline, baseline_pairs)
print(F2.shape)
print("main features:", len(main_features()))
print("missing values:", F2[main_features()].isna().sum().sum())
print(F2["label"].value_counts())
print("flood:", F2["flood"].sum(), "  other attack:", (F2["label"] == 1).sum() - F2["flood"].sum())
print("hour blocks:", F2["time_block"].nunique())

PROC = Path("../../data/processed/device_telemetry")
PROC.mkdir(parents=True, exist_ok=True)
F2.to_parquet(PROC / "phase2_features.parquet")

(12443, 53)
main features: 45
missing values: 0
label
1    7161
0    5282
Name: count, dtype: int64
flood: 7056   other attack: 105
hour blocks: 73
